# Islamic Audiobook Generator
### Run each cell in order. Runtime > Change runtime type > T4 GPU

In [ ]:
!pip install numpy==1.26.4
!pip install "librosa>=0.11.0" "s3tokenizer" "torch>=2.6.0" "torchaudio>=2.6.0" "transformers>=4.46.3" "diffusers>=0.29.0" "resemble-perth>=1.0.1" "conformer>=0.3.2" "safetensors>=0.5.3" "omegaconf" "gradio" "pymupdf" "pydub"
!pip install chatterbox-tts --no-deps

In [ ]:
!nvidia-smi

In [ ]:
import os, uuid, fitz, torchaudio as ta, gradio as gr
from pydub import AudioSegment
from chatterbox.tts import ChatterboxTTS

os.makedirs('outputs', exist_ok=True)
model = ChatterboxTTS.from_pretrained(device='cuda')

def extract_text(pdf_path):
    text = ''
    for page in fitz.open(pdf_path):
        text += page.get_text('text') or ''
    return text

def chunk_text(text, max_chars=200):
    sentences = text.replace('\n', ' ').split('. ')
    chunks, current = [], ''
    for s in sentences:
        if len(current) + len(s) < max_chars:
            current += s + '. '
        else:
            if current:
                chunks.append(current.strip())
            current = s + '. '
    if current:
        chunks.append(current.strip())
    return chunks

def process(text, voice_file):
    job_id = str(uuid.uuid4())
    output_path = f'outputs/{job_id}_audiobook.mp3'
    if not text.strip():
        return None, 'No text provided'
    chunks = chunk_text(text)
    audio_segments = []
    for i, chunk in enumerate(chunks):
        chunk_path = f'outputs/{job_id}_chunk_{i}.wav'
        wav = model.generate(chunk, audio_prompt_path=voice_file)
        ta.save(chunk_path, wav, model.sr)
        audio_segments.append(AudioSegment.from_wav(chunk_path))
    combined = sum(audio_segments)
    combined.export(output_path, format='mp3')
    for i in range(len(chunks)):
        os.remove(f'outputs/{job_id}_chunk_{i}.wav')
    return output_path, 'Done!'

def generate_from_pdf(pdf_file, voice_file):
    text = extract_text(pdf_file)
    if not text.strip():
        return None, 'Could not extract text from PDF'
    return process(text, voice_file)

def generate_from_text(text, voice_file):
    return process(text, voice_file)

with gr.Blocks(title='Islamic Audiobook Generator') as demo:
    gr.Markdown('# Islamic Audiobook Generator')
    with gr.Tab('PDF Mode'):
        pdf_input = gr.File(label='Upload Islamic Book (PDF)', file_types=['.pdf'])
        voice_input1 = gr.File(label='Upload Scholar Voice (WAV)', file_types=['.wav'])
        btn1 = gr.Button('Generate Audiobook', variant='primary')
        audio_out1 = gr.Audio(label='Generated Audiobook', type='filepath')
        status1 = gr.Textbox(label='Status')
        btn1.click(fn=generate_from_pdf, inputs=[pdf_input, voice_input1], outputs=[audio_out1, status1])
    with gr.Tab('Text Mode'):
        text_input = gr.Textbox(label='Paste Text Here', lines=15, placeholder='Paste your Urdu/Arabic text here...')
        voice_input2 = gr.File(label='Upload Scholar Voice (WAV)', file_types=['.wav'])
        btn2 = gr.Button('Generate Audiobook', variant='primary')
        audio_out2 = gr.Audio(label='Generated Audiobook', type='filepath')
        status2 = gr.Textbox(label='Status')
        btn2.click(fn=generate_from_text, inputs=[text_input, voice_input2], outputs=[audio_out2, status2])

demo.launch(share=True)